# Sentinel-1 land-clearing detection over NSW

Sequential omnibus change detection (Conradsen et al. 2016) on dual-pol
Sentinel-1 GRD, run server-side in Google Earth Engine, then constrained to
land clearing by change direction, a baseline woody mask and a minimum
mapping unit.

Runs in Colab or locally. You need a Google Earth Engine account and a
Cloud project with the Earth Engine API enabled.


In [ ]:
# Colab only
# !pip install -q earthengine-api geemap
# !git clone -q https://github.com/JonathanMagson/wind_turbine_detection.git
# %cd wind_turbine_detection/land_clearing_s1


In [ ]:
import ee, geemap

PROJECT = 'your-gee-project'   # <-- change me
ee.Authenticate()
ee.Initialize(project=PROJECT)


## 1. Pick an AOI and a date range

The presets sit in NSW landscapes where SLATS has repeatedly reported woody
clearing. Keep the AOI small (a few hundred km2) for interactive runs.


In [ ]:
import nsw, omnibus, s1

AOI_NAME = 'moree'
START, END = '2023-01-01', '2024-01-01'
ORBIT_PASS = 'DESCENDING'

aoi = nsw.aoi_geometry(name=AOI_NAME)
aoi.bounds().getInfo()['coordinates']


## 2. Build the time series

One relative orbit only, converted from dB to linear power and scaled by the
equivalent number of looks. `build_series` picks the relative orbit with the
most acquisitions fully containing the AOI.


In [ ]:
im_list, dates, rel_orbit = s1.build_series(
    aoi, START, END, orbit_pass=ORBIT_PASS, stride=1)

n_intervals = len(dates) - 1
print(f'{len(dates)} acquisitions on relative orbit {rel_orbit}')
print(dates)


## 3. Run the omnibus test

`alpha` is the per-test significance level. 0.01 is the tutorial default;
lower it (1e-3, 1e-4) if the change maps look speckly. `median=True` applies
a 5x5 median filter, which helps on noisy series at the cost of edge detail.


In [ ]:
ALPHA = 0.01

result = ee.Dictionary(omnibus.change_maps(im_list, median=False, alpha=ALPHA))
bmap = ee.Image(result.get('bmap'))   # one band per interval
cmap = ee.Image(result.get('cmap'))   # most recent change
smap = ee.Image(result.get('smap'))   # first change
fmap = ee.Image(result.get('fmap'))   # number of changes


## 4. Constrain to land clearing

Negative-definite changes only (`bmap == 2`, i.e. backscatter dropped),
inside a baseline woody mask, above a 0.5 ha minimum mapping unit.


In [ ]:
WOODY = 'worldcover'   # 'worldcover' | 'hansen' | 'palsar' | 'none'
MMU_HA = 0.5

mask = nsw.woody_mask(WOODY)
clearing = nsw.clearing_from_bmap(bmap, n_intervals, mask=mask,
                                  min_mmu_ha=MMU_HA, scale=10)

total_ha = nsw.cleared_area_ha(clearing, aoi, scale=10)
print(f'detected clearing: {total_ha:.1f} ha')


## 5. Map it


In [ ]:
m = geemap.Map()
m.centerObject(aoi, 11)

s1_vis = {'min': -25, 'max': 0, 'bands': ['VH']}
first_img = (ee.ImageCollection('COPERNICUS/S1_GRD')
             .filterBounds(aoi).filterDate(START, END)
             .filter(ee.Filter.eq('relativeOrbitNumber_start', rel_orbit))
             .sort('system:time_start').first())
m.addLayer(first_img.clip(aoi), s1_vis, 'S1 VH (first date)')
m.addLayer(mask.selfMask().clip(aoi), {'palette': ['228B22']}, 'woody baseline', False)
m.addLayer(clearing.select('first_interval').selfMask().clip(aoi),
           {'min': 1, 'max': n_intervals, 'palette': ['fee5d9','fb6a4a','a50f15']},
           'clearing (earlier = pale)')
m


## 6. When did it happen

Cleared hectares per acquisition interval. Land clearing shows up as a spike
in one or two adjacent intervals; a flat spread across all intervals usually
means the woody mask or MMU is too permissive.


In [ ]:
per_interval = nsw.area_by_interval(clearing, aoi, n_intervals, scale=10)
for i in range(1, n_intervals + 1):
    ha = per_interval.get(i, 0.0)
    if ha > 0:
        print(f'{dates[i-1]} -> {dates[i]}: {ha:8.2f} ha')


## 7. Validate

Hansen GFC is the zero-setup reference, but it is tuned to closed-canopy
forest and will under-report NSW clearing in sparse woody vegetation. For a
defensible number, download the SLATS woody vegetation change layer for the
matching year from the NSW SEED portal, upload it as a GEE table asset, and
use `validate.py --reference asset`.

```
python validate.py --detected projects/<proj>/assets/nsw_clearing_moree_2023-01-01_2024-01-01 \
    --reference asset --reference-asset projects/<proj>/assets/slats_2023_moree --aoi moree
```


In [ ]:
import validate

ref = validate.hansen_reference(2023, aoi)
counts = validate.confusion(clearing, ref, aoi, scale=30)
print(counts)
print(validate.metrics(counts))


## 8. Export

`detect.py` does all of the above from the command line and can start a
Drive or asset export:

```
python detect.py --aoi moree --start 2023-01-01 --end 2024-01-01 \
    --project your-gee-project --export drive
```
